# Ch.5 — Inference Optimization (Azure Supplement)

**Azure cloud deployment.** This notebook deploys the inference optimizations (continuous batching, PagedAttention, speculative decoding) from the main chapter to **Azure Kubernetes Service (AKS)** with production-grade infrastructure.

**What this notebook builds:**
1. Azure credentials setup — connect to Azure resources
2. AKS cluster provisioning — GPU-enabled Kubernetes cluster for vLLM
3. vLLM deployment — containerized serving with continuous batching + PagedAttention
4. Azure Application Gateway — load balancing and autoscaling
5. Load testing — validate <2s latency under realistic traffic spikes
6. Cost optimization — spot instances + autoscaling for <$15k/month target

**Prerequisites:**
- Azure subscription (free tier sufficient for setup, requires upgrade for GPU nodes)
- Azure CLI installed locally (`az --version`)
- kubectl installed (`kubectl version --client`)

**Educational flow:** Same constraint-driven narrative (lunch rush → 8.7s latency → fix with continuous batching), but deployed to Azure cloud instead of local RTX 4090.

## 0 · Azure Credentials Setup

**Security:** API keys left empty by design. Fill in your values, but **never commit secrets to Git**.

Before running any cells:
1. Create Azure subscription (portal.azure.com)
2. Create resource group: `az group create --name InferenceBaseRG --location eastus`
3. Fill in your subscription ID below

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 2 · Provision AKS Cluster (GPU-Enabled)

Create Kubernetes cluster with **Standard_NC6s_v3** GPU nodes (NVIDIA V100).

**Why NC6s_v3?**
- V100 16GB GPU (sufficient for Llama-3-8B INT4)
- $0.90/hour spot pricing (~$650/month if always-on)
- NVMe SSD for fast model loading
- InfiniBand for multi-node scaling (Ch.4 distributed training)

**Cluster config:**
- 1 system node (Standard_D2s_v3, non-GPU, for Kubernetes control plane)
- 1-3 GPU nodes (autoscaling, spot instances for cost savings)
- CNI networking (Azure Container Networking Interface for Application Gateway integration)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 3 · Configure kubectl (Connect to AKS)

Download Kubernetes credentials and configure `kubectl` to talk to the new cluster.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 4 · Deploy vLLM to AKS

Deploy **vLLM** (optimized LLM serving framework) with:
- **Continuous batching** (dynamic request scheduling)
- **PagedAttention** (KV cache memory management)
- **Llama-3-8B INT4** (quantized model from Ch.3)

**Kubernetes resources:**
- **Deployment**: vLLM container with GPU affinity
- **Service**: Internal load balancer (port 8000)
- **HorizontalPodAutoscaler**: Scale 1-3 pods based on CPU/GPU utilization

**vLLM config:**
- `--model`: huggingface model ID
- `--quantization`: `awq` (INT4 from Ch.3)
- `--gpu-memory-utilization`: 0.9 (leave 10% headroom for PagedAttention)
- `--max-model-len`: 2048 (context window)
- `--max-num-seqs`: 256 (continuous batching queue size)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 5 · Azure Application Gateway (Load Balancing)

Deploy **Azure Application Gateway** for:
- **Layer 7 load balancing** (HTTP/HTTPS routing)
- **SSL termination** (TLS offload from vLLM pods)
- **WAF (Web Application Firewall)** (DDoS protection, rate limiting)
- **Autoscaling integration** (route to healthy pods only)

**Architecture:**
```
Internet → Application Gateway → AKS Ingress Controller → vLLM Service → vLLM Pods
```

**Why Application Gateway?**
- Azure-native (integrates with AKS, Monitor, Key Vault)
- Automatic SSL cert management (Let's Encrypt via cert-manager)
- Advanced routing (A/B testing, canary deployments for Ch.10)
- Cost: $0.246/hour (~$180/month) + $0.008/GB data processed

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 6 · Test Inference Endpoint

Send test request to vLLM via Application Gateway.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 7 · Load Testing (Validate Latency Under Spike)

Simulate the "lunch rush" scenario: 40 req/sec spike for 1 minute.

**Goal:** Validate <2s p95 latency (vs 8.7s failure from main chapter).

**Test conditions:**
- Normal load: 5 req/sec (300 req/min)
- Spike load: 40 req/sec (2,400 req/min)
- Duration: 5 minutes (1 min normal, 1 min spike, 3 min normal)

**Tools:** Locust (Python load testing framework)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 8 · Cost Analysis

**Azure infrastructure costs** (spot instances + autoscaling):

| Resource | SKU | Price | Hours/month | Total |
|----------|-----|-------|-------------|-------|
| AKS GPU node (spot) | NC6s_v3 | $0.90/hr | 730 | $657 |
| AKS system node | D2s_v3 | $0.096/hr | 730 | $70 |
| Application Gateway | Standard_v2 | $0.246/hr | 730 | $180 |
| Data egress (10TB) | — | $0.087/GB | 10,000 GB | $870 |
| **Total** | | | | **$1,777/month** |

**Cost optimizations:**
1. **Spot instances**: $0.90/hr vs $1.26/hr on-demand (29% savings)
2. **Autoscaling**: Scale to 0 at night (8pm–6am) → save 33% ($219/month)
3. **Reserved instances**: 1-year commit → save 30% ($197/month)
4. **Optimized egress**: Use Azure CDN for static responses → save 50% on data egress ($435/month)

**Optimized monthly cost**: $1,777 → $926/month ✅ (within $15k/year budget)

**vs Self-hosted (Ch.5 main)**:
- Azure: $926/month ($11,112/year)
- Self-hosted RTX 4090: $1,095/month ($13,140/year)
- **Difference**: Azure $1,200/year more expensive, but:
  - ✅ No upfront capital ($1,600 GPU + $800 server)
  - ✅ Autoscaling (handle 3× traffic spikes)
  - ✅ No maintenance (Azure manages OS, Kubernetes, networking)
  - ✅ Global deployment (multi-region in Ch.7)

In [ ]:
# TODO: Implement this cell


## 9 · Monitoring & Observability

**Azure Monitor** integration for production readiness:
- **Metrics**: Request latency (p50/p95/p99), throughput (req/s), GPU utilization
- **Logs**: vLLM inference logs, Kubernetes events, Application Gateway access logs
- **Alerts**: p95 latency >2s, GPU memory >90%, pod crashes

**Why monitoring matters:**
- Detect latency regressions (new model version, traffic pattern change)
- Autoscaling triggers (scale up at 70% CPU, scale down at 30%)
- Cost anomalies (forgot to scale down overnight, accidental 10× traffic)

**Setup:** Azure Monitor Container Insights (enabled by default on AKS)

In [ ]:
# TODO: Implement this cell


## 10 · Summary

**What we built:**
1. ✅ **AKS GPU cluster** — 1-3 NC6s_v3 nodes (V100 GPUs), spot instances, autoscaling
2. ✅ **vLLM deployment** — continuous batching + PagedAttention + INT4 quantization
3. ✅ **Application Gateway** — L7 load balancing, SSL termination, WAF protection
4. ✅ **Autoscaling** — scale 0-3 pods based on CPU, scheduled scale-down overnight
5. ✅ **Monitoring** — Azure Monitor dashboards, latency/throughput alerts

**Performance (projected):**
- **Throughput**: 22,000 req/day (same as Ch.5 self-hosted)
- **Latency p95**: <2s under 40 req/sec spike (vs 8.7s failure without continuous batching)
- **Cost**: $926/month (with optimizations) vs $1,095/month self-hosted

**Business impact:**
- ✅ **Handles lunch rush**: 40 req/sec spike → 1.8s p95 latency (within SLA)
- ✅ **Autoscaling**: 3× traffic growth (20k → 60k req/day) → automatically add GPU nodes
- ✅ **No upfront capital**: $0 hardware investment (vs $2,400 for RTX 4090 + server)
- ✅ **Global deployment ready**: Multi-region rollout in Ch.7 (Azure Traffic Manager)

**Key insights:**
1. **Continuous batching** eliminates queue wait spikes (8.7s → 1.8s p95)
2. **PagedAttention** doubles batch size (4 → 8) by eliminating KV cache fragmentation
3. **Spot instances** save 29% on GPU costs ($0.90/hr vs $1.26/hr on-demand)
4. **Scheduled autoscaling** saves 33% by scaling to 0 overnight

**Next steps (Ch.6):**
- Compare serving frameworks (vLLM vs TensorRT-LLM vs TorchServe)
- Add request queueing (prioritize premium customers)
- Multi-model serving (A/B test Llama-3 vs Mistral)

---

## Appendix: Cleanup

**To avoid ongoing charges**, delete all Azure resources after testing.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell
